# Preprocessing

Convert KaVE IDE logs into canonical events and coding sessions.


## 1. Input data

Set the KaVE release directory in `configs/paths.json`. The parser reads JSON events from the participant ZIP archives.


In [ ]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
while not (_root / 'project_paths.py').is_file():
    if _root == _root.parent:
        raise RuntimeError('Open this notebook from within the ide-task-inference repository.')
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from project_paths import get_paths, export_discovery_results
paths = get_paths()

import json
import subprocess
CODE_DIR = paths.root
DATASET_ROOT = paths.raw_data
OUTPUT_DIR = paths.preprocessing
print('Raw dataset:', DATASET_ROOT)
print('Intermediate output:', OUTPUT_DIR)


## 2. Existing outputs


In [ ]:
stats_path = OUTPUT_DIR / 'preprocessing_stats.json'
events_path = OUTPUT_DIR / 'canonical_events.jsonl'
sessions_path = OUTPUT_DIR / 'sessions.jsonl'

for path in [stats_path, events_path, sessions_path]:
    if path.exists():
        print(f'{path.name}: {path.stat().st_size / 1024**2:.2f} MB')
    else:
        print(f'{path.name}: missing')

if stats_path.exists():
    stats = json.loads(stats_path.read_text(encoding='utf-8'))
    print('\nPipeline:', stats.get('pipeline'))
    print('Total parsed:', f"{stats.get('total_parsed', 0):,}")
    print('Total normalized:', f"{stats.get('total_normalized', 0):,}")
    print('Total sessions:', f"{stats.get('session_stats', {}).get('total_sessions', 0):,}")
    print('Config:', stats.get('config'))


## 3. Check a small sample


In [ ]:
from pipeline.preprocessing.kave_parser import KaVEParser
from pipeline.preprocessing.kave_normalizer import KaVENormalizer

if DATASET_ROOT.exists():
    parser = KaVEParser(DATASET_ROOT)
    normalizer = KaVENormalizer()
    zip_paths = parser.discover_zip_files()
    print('ZIP archives discovered:', len(zip_paths))
    sample_raw = next(parser.iter_events(zip_paths=zip_paths[:1]))
    sample_event = normalizer.normalize(sample_raw)
    print('Raw KaVE type:', sample_raw.get('kave_type'))
    print('Canonical type:', sample_event.canonical_type.value)
    print('Session id:', sample_event.session_id)
    print('Timestamp:', sample_event.timestamp)
else:
    print('Dataset root does not exist. Update DATASET_ROOT before running parser checks.')


## 4. Run preprocessing

Set `RUN_PIPELINE = True` to process the dataset. Use `MAX_ZIPS` for a shorter run.


In [ ]:
RUN_PIPELINE = False
MAX_ZIPS = None  # Use a small integer such as 1 for smoke tests. None means full dataset.
INACTIVITY_MIN = 15
MIN_SESSION_EVENTS = 10

cmd = [
    sys.executable, '-m', 'pipeline.preprocessing.run_preprocessing',
    '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--inactivity-min', str(INACTIVITY_MIN),
    '--min-session-events', str(MIN_SESSION_EVENTS),
]
if MAX_ZIPS is not None:
    cmd.extend(['--max-zips', str(MAX_ZIPS)])

print('Command:')
print(' '.join(cmd))

if RUN_PIPELINE:
    subprocess.run(cmd, cwd=CODE_DIR, check=True)
else:
    print('Pipeline execution skipped. Set RUN_PIPELINE=True to run it.')


## 5. Inspect the saved sessions


In [ ]:
if sessions_path.exists():
    with sessions_path.open(encoding='utf-8') as f:
        for idx, line in zip(range(3), f):
            session = json.loads(line)
            first_type = session[0].get('canonical_type') if session else None
            first_time = session[0].get('timestamp') if session else None
            print(f'session {idx}: events={len(session):,}, first_type={first_type}, first_timestamp={first_time}')
else:
    print('sessions.jsonl does not exist yet.')


## 6. Output files

- `canonical_events.jsonl` contains one event per line.
- `sessions.jsonl` contains one list of events per line. Task Discovery reads this file.
- `preprocessing_stats.json` records the run settings and dataset statistics.
